# 🫀 Cardiac Shape GNN — Demo Notebook

This notebook walks through the full pipeline:
1. Clone the SSM repo and install dependencies
2. Generate a small synthetic LV dataset
3. Train the CardiacGNN
4. Visualise predictions
5. Run inference on an ACDC patient

> **Tip**: Enable GPU runtime in Colab (`Runtime → Change runtime type → T4 GPU`)

In [ ]:
# ── 1. Clone repos & install deps ──────────────────────────────────────────
!git clone --quiet https://github.com/YOUR_USERNAME/cardiac-shape-gnn.git
!git clone --quiet https://github.com/UK-Digital-Heart-Project/Statistical-Shape-Model.git

import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch-geometric', 'vtk', 'trimesh', 'nibabel',
                'scikit-image', 'tqdm', 'pyyaml'], check=True)

%cd cardiac-shape-gnn
print('✓ Setup complete')

In [ ]:
# ── 2. Generate a small dataset (50 samples for demo) ─────────────────────
!python scripts/generate_dataset.py --config configs/default.yaml --num_samples 50

In [ ]:
# ── 3. Inspect a generated graph ───────────────────────────────────────────
import sys; sys.path.insert(0, 'src')
import numpy as np
import matplotlib.pyplot as plt
from utils.visualization import plot_graph_3d, plot_slice_contours
import json

graph = np.load('data/heart_dataset_gnn/graphs/graph_000.npz')
with open('data/heart_dataset_gnn/slices/slices_000.json') as f:
    slices = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 3D graph
ax3d = fig.add_subplot(1, 2, 1, projection='3d')
plot_graph_3d(graph['nodes'], graph['edges'], graph['node_types'], ax=ax3d)

# Mid-slice
mid = slices['slices'][len(slices['slices']) // 2]
plot_slice_contours(
    np.array(mid['endo_contour']),
    np.array(mid['epi_contour']),
    z=mid['z_position'],
    ax=axes[1],
)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Train for a few epochs ──────────────────────────────────────────────
!python scripts/train.py --config configs/default.yaml --epochs 20

In [ ]:
# ── 5. Visualise reconstruction on validation sample ──────────────────────
import torch
from torch_geometric.data import Data, Batch
from models.cardiac_gnn import CardiacGNN
from training.trainer import Trainer
from utils.visualization import plot_prediction_vs_truth

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CardiacGNN().to(device)
Trainer.load_checkpoint(model, 'outputs/checkpoints/best_model.pth', device)
model.eval()

g = np.load('data/heart_dataset_gnn/graphs/graph_010.npz')
data = Data(
    x=torch.tensor(g['nodes'], dtype=torch.float32),
    edge_index=torch.tensor(g['edges'].T, dtype=torch.long)
)
batch = Batch.from_data_list([data]).to(device)

with torch.no_grad():
    pred = model(batch).cpu().numpy()

gt = g['nodes'][:, :3]
plot_prediction_vs_truth(gt, pred, title='Demo Reconstruction')

## ACDC Inference
Upload an ACDC `*_gt.nii.gz` file and update the path below.

In [ ]:
# ── 6. ACDC inference (update path below) ─────────────────────────────────
LABEL_PATH = 'path/to/patient101_frame01_gt.nii.gz'

from data.acdc_loader import acdc_patient_to_graph

graph_dict = acdc_patient_to_graph(label_path=LABEL_PATH)
nodes = torch.tensor(graph_dict['nodes'], dtype=torch.float32)
edges = torch.tensor(graph_dict['edges'].T, dtype=torch.long)
data = Data(x=nodes, edge_index=edges)
batch = Batch.from_data_list([data]).to(device)

with torch.no_grad():
    pred_acdc = model(batch).cpu().numpy()

plot_prediction_vs_truth(
    graph_dict['nodes'][:, :3], pred_acdc,
    title='ACDC Patient Prediction'
)